# Lab 3: Why Classical Machine Learning

## Objective:
The objective of this lab is to:

1. Understand **why classical machine learning methods** are used in text similarity and document retrieval tasks.
2. Study and implement three core similarity techniques:
   - **Jaccard Similarity** - set-based word overlap
   - **Cosine Similarity** - vector angle-based comparison
   - **TF-IDF (Term Frequency - Inverse Document Frequency)** - weighted word importance
3. Understand the **limitations** of each method and how each newer method overcomes the shortcomings of the previous one.
4. Apply these methods to a research paper recommendation system.

## Introduction:
Measuring how similar two pieces of text are is a core problem in machine learning. We will walk through the progression of algorithms, starting from simple set-based similarity (Jaccard), moving to coordinate-based vector similarity (Cosine), and finally introducing statistical weighting (TF-IDF) and efficient retrieval structures (KNN) to handle large-scale data.

### 1. Jaccard Similarity

Jaccard Similarity measures the similarity between two sets of data by dividing the number of shared features (intersection) by the total number of unique features (union). 

**Mathematical Formula:**
$$JS(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

**Example Application:**
Let's evaluate two document titles (referred to as "Papers" on the whiteboard):
* **Paper 1:** "Vision AI Vision" $\rightarrow$ Unique words: {vision, AI} (Size = 2)
* **Paper 2:** "NLP AI" $\rightarrow$ Unique words: {NLP, AI} (Size = 2)

**Calculation:**
* **Intersection ($\cap$):** {AI} $\rightarrow$ Size = 1
* **Union ($\cup$):** {vision, AI, NLP} $\rightarrow$ Size = 3
* **Result:** $JS(\text{Paper 1, Paper 2}) = \frac{1}{3} \approx 0.33$

**Limitation:**
Jaccard Similarity strictly relies on unique sets. It **ignores the count or frequency** of words. In Paper 1, the word "Vision" appears twice, but Jaccard Similarity fails to capture this repeated weight, tracking only its mere presence.

def jaccard_similarity(doc1, doc2):
    # Convert documents to sets of words (lowercased)
    set1 = set(doc1.lower().split())
    set2 = set(doc2.lower().split())
    
    # Calculate intersection and union
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    # Calculate Jaccard Similarity
    js = len(intersection) / len(union)
    
    print(f"Set 1: {set1}")
    print(f"Set 2: {set2}")
    print(f"Intersection: {len(intersection)}, Union: {len(union)}")
    print(f"Jaccard Similarity: {js:.2f}\n")
    return js

paper1 = "Vision AI Vision"
paper2 = "NLP AI"

js_score = jaccard_similarity(paper1, paper2)

### 2. Cosine Similarity

To solve the limitations of Jaccard Similarity, we represent documents as vectors in a multi-dimensional coordinate system. Cosine Similarity calculates the cosine of the angle between these vectors, allowing us to factor in word frequency while remaining resilient to document length.

**Mathematical Formula:**
$$CS = \frac{\vec{A} \cdot \vec{B}}{|\vec{A}| |\vec{B}|}$$

Where the dot product is $\vec{A} \cdot \vec{B} = a_1b_1 + a_2b_2 + \dots + a_nb_n$

**Example Application:**
Using our vocabulary {Vision, AI, NLP} as a 3D vector space $[x, y, z]$:
* **Paper 1:** "Vision AI Vision" $\Rightarrow \vec{A} = [2, 1, 0]$ (Vision occurs twice)
* **Paper 2:** "NLP AI" $\Rightarrow \vec{B} = [0, 1, 1]$

**Calculation:**
* **Dot Product ($\vec{A} \cdot \vec{B}$):** $(2 \times 0) + (1 \times 1) + (0 \times 1) = 1$
* **Magnitude of A ($|\vec{A}|$):** $\sqrt{2^2 + 1^2 + 0^2} = \sqrt{5} \approx 2.236$
* **Magnitude of B ($|\vec{B}|$):** $\sqrt{0^2 + 1^2 + 1^2} = \sqrt{2} \approx 1.414$
* **Result:** $CS = \frac{1}{2.236 \times 1.414} = \frac{1}{3.162} \approx 0.316$

**Limitation:**
While it improves upon Jaccard by accounting for word frequency, Cosine Similarity **treats all words equally**. It lacks statistical weight. Common words like "for" or "the" will skew the vector just as much as important domain words like "Transformer" or "NLP".

import numpy as np

def cosine_similarity(vec_a, vec_b):
    # Convert lists to numpy arrays
    A = np.array(vec_a)
    B = np.array(vec_b)
    
    # Calculate dot product and magnitudes
    dot_product = np.dot(A, B)
    mag_A = np.linalg.norm(A)
    mag_B = np.linalg.norm(B)
    
    # Calculate Cosine Similarity
    cs = dot_product / (mag_A * mag_B)
    
    print(f"Vector A: {A}, Magnitude: {mag_A:.3f}")
    print(f"Vector B: {B}, Magnitude: {mag_B:.3f}")
    print(f"Dot Product: {dot_product}")
    print(f"Cosine Similarity: {cs:.3f}\n")
    return cs

### Vocab columns: [Vision, AI, NLP]
vector_paper1 = [2, 1, 0]
vector_paper2 = [0, 1, 1]

cs_score = cosine_similarity(vector_paper1, vector_paper2)

## 3. TF-IDF (Term Frequency - Inverse Document Frequency)

To reduce the error margin of Cosine Similarity, we apply **TF-IDF**. This technique assigns a statistical weight to words, penalizing highly frequent "stop words" and rewarding rare, meaningful words. Using natural log ($\ln$) helps scale the values and reduces computation time.

**Mathematical Formula for IDF:**
$$IDF = \ln\left(\frac{N}{df}\right)$$
Where $N$ is total documents and $df$ is the document frequency (how many documents contain the word).

**Corpus Example:**
1. Transformer for NLP
2. Transformer for CV
3. CNNs for CV

**IDF Calculations ($N=3$):**
* `for`: $\ln(3/3) = 0$ *(Common word neutralized)*
* `Transformer`: $\ln(3/2) \approx 0.40$
* `CV`: $\ln(3/2) \approx 0.40$
* `NLP`: $\ln(3/1) \approx 1.10$ *(Rare word prioritized)*
* `CNNs`: $\ln(3/1) \approx 1.10$

### Limitation of TF-IDF

> **Problem: It still treats each word as independent — it has no understanding of meaning.**

- `car` and `automobile` are treated as **completely different words**, even though they mean the same thing.
- `bank (finance)` and `bank (river)` are treated as **the same word**, even though they mean different things.
- TF-IDF cannot understand **context**, **synonyms**, or **word relationships**.

This limitation eventually motivated the development of **word embeddings** (Word2Vec, GloVe) and then **Transformers** (BERT, GPT), which learn dense vector representations capturing semantic meaning.

### Transitioning to K-Nearest Neighbors (KNN)
While TF-IDF vectors give us excellent contextual similarity, using simple `for loops` to calculate pairwise Cosine Similarity is highly inefficient for large text corpora. To solve this, we map our TF-IDF vectors into a **K-Nearest Neighbors (KNN)** structural tree (like a KD-Tree or Ball Tree). This allows the system to partition the coordinate space and find the closest documents logarithmically rather than linearly, making it suitable for very large files.

### 4) K-Nearest Neighbors (KNN)


K-Nearest Neighbors (KNN) is a **supervised machine learning algorithm** used for both **classification** and **regression**. It classifies a data point based on how its neighbors are classified.



#### Key Idea
- Similar data points are located close to each other.
- Distance or similarity measures are used to identify neighbors.



#### Steps of KNN Algorithm
1. Choose the number of neighbors (**K**)
2. Calculate distance/similarity between query point and all training data
3. Select the **K nearest neighbors**
4. Perform:
   - **Classification:** Majority voting
   - **Regression:** Average of values



#### Distance Measures

#### Euclidean Distance
$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}
$$

#### Cosine Similarity
$$
CS = \frac{A \cdot B}{|A| |B|}
$$



#### Example
- Paper 1 → [2, 1, 0]  
- Paper 2 → [0, 1, 1]  

Steps:
- Compute similarity
- Choose K nearest documents
- Assign label based on majority



### Choosing K
- Small K → Sensitive to noise  
- Large K → More stable but less precise  
- Common values: **K = 3, 5, 7**



#### Advantages
- Simple and easy to implement  
- No training phase (lazy learning)  
- Works well for small datasets  



#### Limitations
- Slow for large datasets  
- Sensitive to irrelevant features  
- Requires feature scaling  



#### Connection with Cosine Similarity
- Instead of Euclidean distance, KNN can use **cosine similarity**
- Higher cosine similarity → closer neighbor
- Useful in **text classification problems**

import math
from collections import Counter

corpus = [
    "Transformer for NLP",
    "Transformer for CV",
    "CNNs for CV"
]

def calculate_idf(corpus):
    N = len(corpus)
    # Tokenize corpus into unique sets per document
    tokenized_docs = [set(doc.split()) for doc in corpus]
    
    # Get overall vocabulary
    vocab = set(word for doc in tokenized_docs for word in doc)
    
    idf_scores = {}
    for word in vocab:
        # Count how many documents contain the word
        df = sum(1 for doc in tokenized_docs if word in doc)
        # Calculate natural log IDF (using math.log)
        idf_scores[word] = math.log(N / df)
        
    return idf_scores

idf_dict = calculate_idf(corpus)

print("IDF Scores calculated using Natural Log:")
for word, score in sorted(idf_dict.items(), key=lambda item: item[1]):
    print(f"'{word}': {score:.2f}")

In [5]:


import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# 1. Synthetic Research Dataset
# -----------------------------
paper_titles = [
    ("ResNet-3D: Residual Grids for Spatial Volumetric Analysis", "CV", "CNNs", "Dr_Y_LeCun"),
    ("ViT-Scale: Scaling Vision Transformers for Dense Prediction", "CV", "Transformers", "Dr_A_Vaswani"),
    ("SimCLR-V3: Contrastive Frameworks for Self-Supervised Vision", "CV", "CL", "Dr_G_Hinton"),
    ("Masked Autoencoders Are Scalable Vision Visualizers", "CV", "Transformers", "Dr_Y_LeCun"),
    ("NeRF-Graph: Neural Radiance Fields on Graph Topologies", "CV", "GNNs", "Dr_J_Leskovec"),
    ("Real-Time Semantic Segmentation via Dilated Spatial Kernels", "CV", "CNNs", "Dr_Y_LeCun"),

    ("BERT-Large: Pre-training of Deep Bidirectional Transformers", "NLP", "Transformers", "Dr_A_Vaswani"),
    ("GPT-Next: Autoregressive Language Modeling at Scale", "NLP", "Transformers", "Dr_A_Vaswani"),
    ("Contrastive Sentence Embeddings via Semantic Invariance", "NLP", "CL", "Dr_Y_Bengio"),
    ("Long-Short Sequence Parsing via Linear Attention Windows", "NLP", "Transformers", "Dr_A_Vaswani"),
    ("Text-Graph Recurrent Transformers for Structured Document Analysis", "NLP", "GNNs", "Dr_J_Leskovec"),

    ("GCN-V2: Scalable Graph Convolutional Networks via Node Sampling", "Graph_AI", "GNNs", "Dr_J_Leskovec"),
    ("Graph Attention Networks with Multi-Head Structural Alignment", "Graph_AI", "Transformers", "Dr_A_Vaswani"),
    ("Self-Supervised Graph Contrastive Learning via Subgraph Masking", "Graph_AI", "CL", "Dr_Y_Bengio"),
    ("Temporal Graph Networks for Dynamic Relational Interaction Stream", "Graph_AI", "GNNs", "Dr_J_Leskovec"),
]

# -----------------------------
# 2. Convert papers → text docs
# -----------------------------
paper_texts = [
    f"{title} {domain} {method} {author}"
    for title, domain, method, author in paper_titles
]

N = len(paper_texts)  # total number of documents

# 3. TF-IDF
# Step 3a: Get raw term counts (TF numerator)
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(paper_texts).toarray()  # shape: (N, V)
vocab = count_vectorizer.get_feature_names_out()

# Step 3b: Term Frequency (TF) — log normalization
#   TF(t, d) = log(1 + count(t, d))
#   Dampens the effect of high raw counts
tf_matrix = np.log1p(count_matrix)  # log(1 + count), shape: (N, V)

# Step 3c: Inverse Document Frequency (IDF) — log-smoothed
#   DF(t) = number of docs containing term t
#   IDF(t) = log((1 + N) / (1 + DF(t))) + 1   ← sklearn's smooth IDF formula
#   The +1 outside prevents zero IDF for very common terms
df = np.count_nonzero(count_matrix, axis=0)          # shape: (V,)
idf_vector = np.log((1 + N) / (1 + df)) + 1          # shape: (V,)

# Step 3d: TF-IDF = TF × IDF (element-wise broadcast)
tfidf_matrix = tf_matrix * idf_vector                 # shape: (N, V)

# Step 3e: L2-normalize each document row so dot product = cosine similarity
row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
row_norms[row_norms == 0] = 1                         # avoid division by zero
tfidf_matrix_norm = tfidf_matrix / row_norms          # shape: (N, V)

# 4. Cosine Similarity 

# Cosine similarity between two vectors u, v:
#   cos(u, v) = (u · v) / (||u|| * ||v||)
# Since tfidf_matrix_norm rows are already L2-normalized:
#   cos(u_norm, v_norm) = u_norm · v_norm  (just a dot product)
cosine_sim_matrix = cosine_similarity(tfidf_matrix_norm)  # shape: (N, N)

# 5. Jaccard Similarity — 
def jaccard_similarity(a: str, b: str) -> float:
    """
    Jaccard(A, B) = |A ∩ B| / |A ∪ B|
    Uses the unique word-sets of two strings.
    """
    set_a = set(a.lower().split())
    set_b = set(b.lower().split())
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


# 6. Recommendation Function
def recommend(paper_index: int, top_k: int = 5) -> None:
    query_title = paper_titles[paper_index][0]
    print(f"\n{'='*70}")
    print(f"Query Paper:  {query_title}")
    print(f"{'='*70}")

    cosine_scores  = []
    tfidf_scores   = []
    jaccard_scores = []

    for i in range(N):
        if i == paper_index:
            continue

        # ── Cosine Similarity (from the normalised TF-IDF vectors) ──
        cos_score = cosine_sim_matrix[paper_index][i]
        cosine_scores.append((i, cos_score))

        # ── TF-IDF Score: dot product of raw (non-normalised) rows ──
        #    Measures how much shared weighted vocabulary the two docs have
        #    without the length-normalisation step that Cosine applies.
        raw_tfidf_score = np.dot(tfidf_matrix[paper_index], tfidf_matrix[i])
        tfidf_scores.append((i, raw_tfidf_score))

        # ── Jaccard Similarity (independent of TF-IDF) ──
        jac_score = jaccard_similarity(paper_texts[paper_index], paper_texts[i])
        jaccard_scores.append((i, jac_score))

    # Sort each ranking independently
    cosine_scores.sort(key=lambda x: x[1], reverse=True)
    tfidf_scores.sort(key=lambda x: x[1], reverse=True)
    jaccard_scores.sort(key=lambda x: x[1], reverse=True)

    # ── Print: Cosine Similarity ──
    print(f"\n{'─'*70}")
    print(" Cosine Similarity  ")
    print(f"{'─'*70}")
    for i, score in cosine_scores[:top_k]:
        print(f"  {paper_titles[i][0]:<58} | {score:.4f}")

    # ── Print: TF-IDF Raw Score ──
    print(f"\n{'─'*70}")
    print(" TF-IDF Score  ")
    print(f"{'─'*70}")
    for i, score in tfidf_scores[:top_k]:
        print(f"  {paper_titles[i][0]:<58} | {score:.4f}")

    # ── Print: Jaccard ──
    print(f"\n{'─'*70}")
    print("Jaccard Similarity  ")
    
    print(f"{'─'*70}")
    for i, score in jaccard_scores[:top_k]:
        print(f"  {paper_titles[i][0]:<58} | {score:.4f}")

# 7. Run example

recommend(0)



Query Paper:  ResNet-3D: Residual Grids for Spatial Volumetric Analysis

──────────────────────────────────────────────────────────────────────
 Cosine Similarity  
──────────────────────────────────────────────────────────────────────
  Real-Time Semantic Segmentation via Dilated Spatial Kernels | 0.2852
  Text-Graph Recurrent Transformers for Structured Document Analysis | 0.1479
  Masked Autoencoders Are Scalable Vision Visualizers        | 0.1261
  ViT-Scale: Scaling Vision Transformers for Dense Prediction | 0.0965
  SimCLR-V3: Contrastive Frameworks for Self-Supervised Vision | 0.0917

──────────────────────────────────────────────────────────────────────
 TF-IDF Score  
──────────────────────────────────────────────────────────────────────
  Real-Time Semantic Segmentation via Dilated Spatial Kernels | 11.2097
  Text-Graph Recurrent Transformers for Structured Document Analysis | 5.3205
  Masked Autoencoders Are Scalable Vision Visualizers        | 4.3390
  ViT-Scale: Scaling V

## Discussion
Throughout this lab, we observed a direct evolutionary line in how classical machine learning handles text similarities:
1.  **Jaccard** acts as a baseline. It is fast and simple but loses valuable context because it only observes binary presence (0 or 1), discarding word frequency completely.
2.  **Cosine Similarity** solved the frequency issue by mapping words to coordinate systems, using the angle between vectors to assess similarity. However, it blindly trusted all words, causing dense clusters of low-value words (like "the", "a", "for") to skew the angles.
3.  **TF-IDF** solved the weighting issue. By pushing term frequencies through a logarithmic inverse document frequency filter, we successfully punished generic words and elevated contextual ones, creating highly accurate semantic vectors.

## Conclusion
Classical machine learning approaches to text require sequential refinement to yield good results. We concluded that raw term frequency is insufficient for accurate similarity matching. By applying TF-IDF to penalize commonality, we generate highly accurate vector representations. Finally, as noted in the lecture, applying these calculations linearly via simple loops is computationally expensive on large datasets; integrating clustering or structural tree algorithms like **KNN** on top of TF-IDF vectors is necessary for deploying these models in real-world, large-scale applications.